# Attention Memory Complexity

Before writing any kernels, we make the memory problem concrete.

Scaled dot-product attention:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

The bottleneck is $QK^T$: a matrix of shape $(S, S)$ per head. At long sequence lengths, this dominates all other memory costs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sequence lengths typical in modern LLMs
seq_lengths = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768, 65536])

# Typical config: 8 heads, d_k=128, batch=1, bfloat16 (2 bytes)
HEADS = 8
D_K = 128
BATCH = 1
BYTES_PER_ELEMENT = 2  # bfloat16

def qkv_bytes(seq, heads=HEADS, d_k=D_K, batch=BATCH, bpe=BYTES_PER_ELEMENT):
    """Q + K + V tensors."""
    return 3 * batch * heads * seq * d_k * bpe

def attn_matrix_bytes(seq, heads=HEADS, batch=BATCH, bpe=BYTES_PER_ELEMENT):
    """Attention matrix: (batch, heads, seq, seq)."""
    return batch * heads * seq * seq * bpe

def output_bytes(seq, heads=HEADS, d_k=D_K, batch=BATCH, bpe=BYTES_PER_ELEMENT):
    """Output O: same shape as Q."""
    return batch * heads * seq * d_k * bpe

def total_naive_bytes(seq):
    return qkv_bytes(seq) + attn_matrix_bytes(seq) + output_bytes(seq)

qkv = np.array([qkv_bytes(s) for s in seq_lengths]) / 1e9
attn = np.array([attn_matrix_bytes(s) for s in seq_lengths]) / 1e9
total = np.array([total_naive_bytes(s) for s in seq_lengths]) / 1e9

print(f"{'seq':>8}  {'QKV (GB)':>10}  {'Attn (GB)':>10}  {'Total (GB)':>11}")
print("-" * 46)
for s, q, a, t in zip(seq_lengths, qkv, attn, total):
    print(f"{s:>8}  {q:>10.3f}  {a:>10.3f}  {t:>11.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(seq_lengths, qkv, 'o-', label='QKV')
ax.plot(seq_lengths, attn, 's-', label='Attention matrix', linewidth=2)
ax.plot(seq_lengths, total, '^--', label='Total', alpha=0.7)
ax.set_xlabel('Sequence length')
ax.set_ylabel('HBM (GB)')
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title('Absolute memory cost')

ax = axes[1]
attn_fraction = attn / total
ax.plot(seq_lengths, attn_fraction, 'o-', color='red')
ax.set_xlabel('Sequence length')
ax.set_ylabel('Attention matrix / total HBM')
ax.set_xscale('log', base=2)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
ax.set_title('Attention matrix fraction of total')

fig.suptitle(f'Naive attention HBM: {HEADS} heads, d_k={D_K}, batch={BATCH}, bfloat16')
fig.tight_layout()
plt.show()

## The quadratic term dominates

At seq=16384:
- QKV: ~0.1 GB (linear in seq)
- Attention matrix: ~4.3 GB (quadratic in seq)

The attention matrix is the problem. It grows as $O(S^2)$ while QKV grows as $O(S)$.

Flash Attention's strategy: never materialize this matrix in HBM. Tile the computation across VMEM so only the active tiles are in fast memory at any time.

The cost: we need to compute the softmax incrementally (online softmax) and recompute the attention weights during the backward pass instead of storing them.

In [ ]:
# Flash attention HBM footprint: O(seq * d), not O(seq^2)
# Q, K, V, O: same as naive
# m, l (log-sum-exp): (batch, heads, seq, 128) float32

def flash_peak_bytes(seq, heads=HEADS, d_k=D_K, batch=BATCH, bpe=BYTES_PER_ELEMENT):
    qkvo = 4 * batch * heads * seq * d_k * bpe
    ml = 2 * batch * heads * seq * 128 * 4  # float32
    return qkvo + ml

flash = np.array([flash_peak_bytes(s) for s in seq_lengths]) / 1e9
savings = total / flash

print(f"{'seq':>8}  {'naive (GB)':>11}  {'flash (GB)':>11}  {'savings':>9}")
print("-" * 46)
for s, n, f, sv in zip(seq_lengths, total, flash, savings):
    print(f"{s:>8}  {n:>11.3f}  {f:>11.3f}  {sv:>9.1f}x")